# Inside the Max Planck Institute for Demographic Research
### DATASCI 530 – Project 2 · Xueqian Cheng

*Key findings will go here once the results are final (3–4 bullets).*

## 1. Introduction

*Project description: what MPIDR is, what the three datasets cover, and the
questions this report answers.*

## 2. Data collection

*Short, non-technical summary of how the data were scraped (see the Project 1
notebook for the code).*

## 3. Changes since Project 1

*Table: what was added or corrected, and why.*

## 4. Data preparation

In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display

pd.set_option("display.max_colwidth", 80)

# Chart style used in every figure
BLUE, GRAY  = "#2a78d6", "#c3c2b7"
INK, INK_2  = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.figsize": (8, 4.5), "figure.dpi": 110,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelcolor": INK_2,
    "axes.edgecolor": GRAY, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "xtick.color": INK_2, "ytick.color": INK_2,
    "legend.frameon": False})

def show_table(df, caption, formats = None):
    """Display a table with a caption, no index and rounded numbers."""
    styler = (df.style.hide(axis = "index")
                .set_caption(caption)
                .format(formats or {}, thousands = ",")
                .set_table_styles([
                    {"selector": "caption",
                     "props": "caption-side: top; font-weight: bold; "
                              "text-align: left; font-size: 1.05em; padding-bottom: 6px;"},
                    {"selector": "th", "props": "text-align: left;"}]))
    display(styler)

**Load the three datasets from Project 1.**

In [2]:
projects = pd.read_csv("mpidr_projects.csv")
members  = pd.read_csv("mpidr_project_members.csv")
pubs_raw = pd.read_csv("mpidr_publications.csv")

# Readable names for the four parts of the institute
LAB_NAMES = {
    "social_demography_myrskylae_11664":                  "Social Demography lab",
    "digital_and_computational_demography_zagheni_11666": "Digital & Computational lab",
    "joint_research_labs_11667":                          "Joint research labs",
    "independent_research_groups_11668":                  "Independent research groups"}
projects["lab"] = projects["unit_group"].map(LAB_NAMES)
pubs_raw["lab"] = pubs_raw["unit_group"].map(LAB_NAMES)

print(len(projects), "projects;", len(members), "project-member rows;",
      len(pubs_raw), "unit-publication rows")

131 projects; 381 project-member rows; 1967 unit-publication rows


**Research keywords and regions.** Both are stored as comma-separated
text, but some of the site's keyword categories contain commas themselves
(for example *Aging, Mortality and Longevity*). Splitting on every comma would
turn one category into several, so these names are protected before splitting.
The only region with a comma is *Korea, South*.

In [3]:
MULTI_PART = ["Aging, Mortality and Longevity",
              "Health Care, Public Health, Medicine, and Epidemiology",
              "International Migration, Ethnic Minorities",
              "Policies, Politics, Administration, Welfare State",
              "Internal Migration, Housing, Urbanisation",
              "Economics, Employment, Retirement"]

def split_keywords(text):
    """Split a keyword string into the site's categories."""
    if pd.isna(text):
        return []
    for name in MULTI_PART:
        text = text.replace(name, name.replace(", ", " | "))
    return [k.replace(" | ", ", ") for k in text.split(", ")]

def split_regions(text):
    if pd.isna(text):
        return []
    return text.replace("Korea, South", "South Korea").split(", ")

# One row per project x keyword, and per project x region
project_keywords = (projects[["project_url", "unit", "lab"]]
                    .assign(keyword = projects["keywords"].map(split_keywords))
                    .explode("keyword").dropna(subset = ["keyword"]))
project_regions  = (projects[["project_url", "unit", "lab"]]
                    .assign(region = projects["regions"].map(split_regions))
                    .explode("region").dropna(subset = ["region"]))

print(project_keywords["keyword"].nunique(), "keyword categories;",
      project_regions["region"].nunique(), "regions")

19 keyword categories; 65 regions


**Publications.** A paper can appear on several units' lists, so each
URL is kept once. The site also often lists a paper twice under different
URLs: first as an MPIDR working paper, then as the published journal article.
For counts of *distinct papers*, these pairs (matched by title) are counted
once, keeping the published version. Each paper's group size is its number of
authors.

In [4]:
# One row per publication URL, with the units that list it
pubs = (pubs_raw.groupby("pub_url", as_index = False)
        .agg(pub_title = ("pub_title", "first"), authors = ("authors", "first"),
             pub_type = ("pub_type", "first"), pub_year = ("pub_year", "first"),
             open_access = ("open_access", "first"), lab = ("lab", "first"),
             n_units = ("unit", "nunique")))
pubs["n_authors"] = pubs["authors"].str.split("; ").str.len()

# Distinct papers: a working paper and its journal version count once
pubs["title_key"] = pubs["pub_title"].str.lower().str.replace(r"[^a-z0-9]", "", regex = True)
pubs["is_working_paper"] = pubs["pub_type"].eq("mpidr_working_papers")
papers = (pubs.sort_values("is_working_paper")
              .drop_duplicates("title_key")
              .drop(columns = ["title_key", "is_working_paper"]))

# Readable publication types
TYPE_NAMES = {"journal_articles": "Journal article", "mpidr_working_papers": "MPIDR working paper",
              "book_chapters": "Book chapter", "preprints": "Preprint",
              "other_papers": "Other paper", "dissertations": "Dissertation",
              "mpidr_technical_reports": "MPIDR technical report", "monographs": "Book",
              "software": "Software", "reports": "Report", "external link": "External link"}
pubs["pub_type"]   = pubs["pub_type"].map(TYPE_NAMES)
papers["pub_type"] = papers["pub_type"].map(TYPE_NAMES)

# Trend charts use complete years only: 2026 was still in progress when
# the data were collected, and the few 2027 items are forthcoming
FIRST_YEAR, LAST_YEAR = 2005, 2025

print(len(pubs), "distinct publication URLs;", len(papers), "distinct papers")

1710 distinct publication URLs; 1606 distinct papers


## 5. Results

### A. How is MPIDR's research organised?

*T1 overview by lab · F1 projects per unit · F2 research topics · F3 regions studied*

### B. Who does the research?

*F4 projects per researcher · F5 group size and project-page publications (optional)*

### C. What does MPIDR publish, and how has it changed?

*T2 publications by type · F6 publications per year · F7 open access over time ·
F8 group size over time*

## 6. Discussion

*Takeaways and limitations.*